In [ ]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-3z3q_cjk
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-3z3q_cjk
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 173e636d9a3cb81c6101a160c1c8c86864eaaf5b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [ ]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <stdint.h> // uint32_t

using element_t = uint32_t;

In [ ]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [ ]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <algorithm>
#include <random>  // for std::mt19937, std::uniform_int_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a1 {

/**
 * Generates and returns a vector of random uniform data of a given length, n,
 * for any integral type. Input range will be [0, 2n].
 */
template < typename T >
std::vector< T > generate_uniform( std::size_t n )
{
    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20240916;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 2 * n);

    std::vector< T > random_data( n ); // init array
    std::generate( std::begin( random_data )
                 , std::end  ( random_data )
                 , [ &rng, &distrib ](){ return static_cast< T >( distrib( rng ) ); });

    return random_data;
}

} // namespace a1
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "algorithm_choices.h"
#pragma once

#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a1 {
namespace cpu {

void run_cpu_baseline( std::vector< element_t > data, std::size_t switch_at, std::size_t n );

} // namespace cpu


namespace gpu {

void run_gpu_soln( std::vector< element_t > data, std::size_t switch_at, std::size_t n );

 void run_gpu_radix_sort( std::vector<element_t> data, std::size_t n);

} // namespace gpu
} // namespace a1
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "cpu_baseline.cu"
/**
 * CPU methods that the GPU should outperform.
 */

#include "algorithm_choices.h"

#include <algorithm> // std::sort()
#include <chrono>    // for timing
#include <iostream>  // std::cout, std::endl

namespace csc485b {
namespace a1      {
namespace cpu     {

/**
 * Simple solution that just sorts the whole array with a built-in sort
 * function and then resorts the last portion in the opposing order with
 * a second call to that same built-in sort function.
 */
void opposing_sort( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    std::sort( data, data + num_elements, std::less< element_t >{} );
    std::sort( data + invert_at_pos, data + num_elements, std::greater< element_t >{} );
}

/**
 * Run the single-threaded CPU baseline that students are supposed to outperform
 * in order to obtain higher grades on this assignment. Times the execution and
 * prints to the standard output (e.g., the screen) that "wall time." Note that
 * the functions takes the input by value so as to not perturb the original data
 * in place.
 */
void run_cpu_baseline( std::vector< element_t > data, std::size_t switch_at, std::size_t n )
{
    auto const cpu_start = std::chrono::high_resolution_clock::now();
    opposing_sort( data.data(), switch_at, n );
    auto const cpu_end = std::chrono::high_resolution_clock::now();

    std::cout << "CPU Baseline time: "
              << std::chrono::duration_cast<std::chrono::nanoseconds>(cpu_end - cpu_start).count()
              << " ns" << std::endl;

    for( auto const x : data ) std::cout << x << " "; std::cout << std::endl;
}

} // namespace cpu
} // namespace a1
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "gpu_solution.cu"
/**
 * GPU solution for CSC 485B Assignment 1
 */

#include "algorithm_choices.h"

#include <algorithm>
#include <chrono>
#include <iostream>
#include <cstdint>

#include "cuda_common.h"

namespace csc485b {
namespace a1 {
namespace gpu {

namespace constants {
    element_t const MAX_VAL = 0xFFFFFFFF;
}

__global__
void warm_the_gpu(
    element_t * data,
    std::size_t invert_at_pos,
    std::size_t num_elements)
{
    std::size_t th_id =
        blockIdx.x * blockDim.x + threadIdx.x;

    if (th_id < num_elements &&
        data[th_id] > num_elements * 100)
    {
        ++data[th_id];
    }
}

__global__
void opposing_sort(element_t* data, std::size_t switch_at, std::size_t num_elements) {
    __shared__ element_t shared[1024];

    std::size_t tid = threadIdx.x;

    std::size_t th_id = blockIdx.x * blockDim.x + tid;

    if (th_id < num_elements) {
        shared[tid] = data[th_id];
    } else {
        shared[tid] = constants::MAX_VAL;
    }

    __syncthreads();

    //alternate ascending descending blocks for global merging of n>=1024
    bool block_ascending = ((blockIdx.x & 1) == 0);

    for (std::size_t k = 2; k <= 1024; k <<= 1) {
        for (std::size_t stride = k >> 1; stride > 0; stride >>= 1) {

            std::size_t partner = tid ^ stride;

            if (partner > tid && partner < 1024) {
                bool ascending = ((tid & k) == 0); // butterfly :D

                if (!block_ascending) {
                    ascending = !ascending;
                }

                element_t a = shared[tid];
                element_t b = shared[partner];

                if ((ascending && a > b) || (!ascending && a < b)) {
                    shared[tid] = b;
                    shared[partner] = a;
                }
            }

            __syncthreads();
        }
    }
    // Flip last n/4 elements in block if n<=1024
    if(num_elements <= 1024) {
        std::size_t suffix_size = num_elements - switch_at;
        std::size_t number_of_pairs = suffix_size / 2;

        if (tid < number_of_pairs) {
            std::size_t left = switch_at + tid;
            std::size_t right = num_elements - 1 - tid;

            element_t temp = shared[left];
            shared[left] = shared[right];
            shared[right] = temp;
        }
    }
    __syncthreads();

    if (th_id < num_elements) {
        data[th_id] = shared[tid];
    }
}

__global__
void reverse_suffix(element_t* data, std::size_t switch_at, std::size_t num_elements) {
    std::size_t th_id = blockIdx.x * blockDim.x + threadIdx.x;

    std::size_t suffix_size = num_elements - switch_at;
    std::size_t number_of_pairs = suffix_size / 2;

    if (th_id >= number_of_pairs) {
        return;
    }

    std::size_t left = switch_at + th_id;
    std::size_t right = num_elements - 1 - th_id;

    element_t temp = data[left];
    data[left] = data[right];
    data[right] = temp;
}

__global__
void bitonic_global_stage(element_t* data, std::size_t num_elements, std::size_t k, std::size_t stride) {
    std::size_t th_id = blockIdx.x * blockDim.x + threadIdx.x;

    if (th_id >= num_elements) {
        return;
    }

    std::size_t partner = th_id ^ stride;

    if (partner > th_id && partner < num_elements) {
        bool ascending = ((th_id & k) == 0);

        element_t a = data[th_id];
        element_t b = data[partner];

        if ((ascending && a > b) || (!ascending && a < b)) {
            data[th_id] = b;
            data[partner] = a;
        }
    }
}

__global__
void build_histograms(const element_t* input, std::size_t n, unsigned int shift, unsigned int* block_histograms) {

    __shared__ unsigned int histogram[256];

    unsigned int tid = threadIdx.x;
    histogram[tid] = 0;

    __syncthreads();

    std::size_t index = blockIdx.x * blockDim.x + tid;

    if (index < n) {
        unsigned int digit = (input[index] >> shift) & 0xFFu;
        atomicAdd(&histogram[digit], 1u);
    }

    __syncthreads();

    block_histograms[blockIdx.x * 256 + tid] = histogram[tid];
}

__global__
void compute_digit_totals(const unsigned int* block_histograms, unsigned int* digit_totals, std::size_t num_blocks) {
    unsigned int digit = threadIdx.x;

    if (digit >= 256) {
        return;
    }

    unsigned int total = 0;

    for (std::size_t b = 0; b < num_blocks; ++b) {
        total += block_histograms[b * 256 + digit];
    }

    digit_totals[digit] = total;
}

__global__
void exclusive_scan(const unsigned int* digit_totals, unsigned int* digit_offsets) {
    __shared__ unsigned int scan[256];

    unsigned int tid = threadIdx.x;

    scan[tid] = digit_totals[tid];
    __syncthreads();

    for (unsigned int offset = 1; offset < 256; offset <<= 1) {
        unsigned int value = 0;

        if (tid >= offset) {
            value = scan[tid - offset];
        }

        __syncthreads();
        scan[tid] += value;
        __syncthreads();
    }

    if (tid == 0) {
        digit_offsets[tid] = 0;
    } else {
        digit_offsets[tid] = scan[tid - 1];
    }

}

__global__
void compute_block_offsets(const unsigned int* block_histograms, const unsigned int* digit_offsets,
                            unsigned int* block_offsets, std::size_t num_blocks) {

    unsigned int digit = threadIdx.x;

    if (digit >= 256) {
        return;
    }

    unsigned int position = digit_offsets[digit];

    for (std::size_t b = 0; b < num_blocks; ++b) {
        block_offsets[b * 256 + digit] = position;
        position += block_histograms[b * 256 + digit];
    }
}

__global__
void radix_scatter(const element_t* input, element_t* output, std::size_t n, unsigned int shift, const unsigned int* block_offsets) {

    unsigned int tid = threadIdx.x;
    std::size_t block_start = blockIdx.x * blockDim.x;
    std::size_t index = block_start + tid;

    if (index >= n) {
        return;
    }

    unsigned int digit = (input[index] >> shift) & 0xFFu;
    unsigned int rank = 0;

    for (unsigned int i = 0; i < tid; ++i) {
        std::size_t previous = block_start + i;

        if (previous < n) {
            unsigned int previous_digit = (input[previous] >> shift) & 0xFFu;

            if (previous_digit == digit) {
                ++rank;
            }
        }
    }

    unsigned int destination = block_offsets[blockIdx.x * 256 + digit] + rank;
    output[destination] = input[index];
}

void calculate_offsets(const unsigned int* d_block_histograms, unsigned int* d_digit_totals,
                       unsigned int* d_digit_offsets, unsigned int* d_block_offsets, std::size_t num_blocks) {

    compute_digit_totals<<<1, 256>>>(d_block_histograms, d_digit_totals, num_blocks);
    exclusive_scan<<<1, 256>>>(d_digit_totals, d_digit_offsets);
    compute_block_offsets<<<1, 256>>>(d_block_histograms, d_digit_offsets, d_block_offsets, num_blocks);
}

void radix_sort(element_t* d_data, element_t* d_temp, std::size_t n) {

    if (n <= 1) {
        return;
    }

    constexpr std::size_t BLOCK_SIZE = 256;
    constexpr std::size_t RADIX_SIZE = 256;

    std::size_t num_blocks = (n + BLOCK_SIZE - 1) / BLOCK_SIZE;

    unsigned int* d_block_histograms;
    unsigned int* d_digit_totals;
    unsigned int* d_digit_offsets;
    unsigned int* d_block_offsets;

    cudaMalloc(&d_block_histograms, num_blocks * RADIX_SIZE * sizeof(unsigned int));
    cudaMalloc(&d_digit_totals, RADIX_SIZE * sizeof(unsigned int));
    cudaMalloc(&d_digit_offsets, RADIX_SIZE * sizeof(unsigned int));
    cudaMalloc(&d_block_offsets, num_blocks * RADIX_SIZE * sizeof(unsigned int));

    element_t* input = d_data;
    element_t* output = d_temp;

    for (unsigned int shift = 0; shift < 32; shift += 8) {

        build_histograms<<<num_blocks, 256>>>(input, n, shift, d_block_histograms);
        calculate_offsets(d_block_histograms, d_digit_totals, d_digit_offsets, d_block_offsets, num_blocks);
        radix_scatter<<<num_blocks, 256>>>(input, output, n, shift, d_block_offsets);
        std::swap(input, output);
    }

    cudaFree(d_block_histograms);
    cudaFree(d_digit_totals);
    cudaFree(d_digit_offsets);
    cudaFree(d_block_offsets);
}

void run_gpu_radix_sort(std::vector<element_t> data, std::size_t n) {
    element_t * d_data;
    element_t * d_temp;

    cudaMalloc((void **) &d_data, sizeof(element_t) * n);
    cudaMalloc((void **) &d_temp, sizeof(element_t) * n);

    cudaMemcpy(d_data, data.data(), sizeof(element_t) * n, cudaMemcpyHostToDevice);

    warm_the_gpu<<<(n + 255) / 256, 256>>>(d_data, 0, n);

    cudaDeviceSynchronize();

    auto const start = std::chrono::high_resolution_clock::now();

    radix_sort(d_data, d_temp, n);

    cudaDeviceSynchronize();

    auto const end = std::chrono::high_resolution_clock::now();

    cudaMemcpy(data.data(), d_data, sizeof(element_t) * n, cudaMemcpyDeviceToHost);

    cudaFree(d_data);
    cudaFree(d_temp);

    std::cout
        << "GPU Radix Sort time: "
        << std::chrono::duration_cast<std::chrono::nanoseconds>(end - start).count()
        << " ns"
        << std::endl;

    for (auto const x : data) {
        std::cout << x << " ";
    }

    std::cout << std::endl;
}

void run_gpu_soln( std::vector< element_t > data, std::size_t switch_at, std::size_t n ) {
    // Kernel launch configurations. Feel free to change these.
    // This is set to maximise the size of a thread block on a T4, but it hasn't
    // been tuned. It's not known if this is optimal.
    std::size_t const threads_per_block = 1024;
    std::size_t const num_blocks =  ( n + threads_per_block - 1 ) / threads_per_block;

    // Allocate arrays on the device/GPU
    element_t * d_data;
    cudaMalloc( (void**) & d_data, sizeof( element_t ) * n );
    CHECK_ERROR("Allocating input array on device");

    // Copy the input from the host to the device/GPU
    cudaMemcpy( d_data, data.data(), sizeof( element_t ) * n, cudaMemcpyHostToDevice );
    CHECK_ERROR("Copying input array to device");

    // Warm the cache on the GPU for a more fair comparison
    warm_the_gpu<<< num_blocks, threads_per_block>>>( d_data, switch_at, n );

    cudaDeviceSynchronize();
    auto const kernel_start = std::chrono::high_resolution_clock::now();

    // Sort elements in each block
    opposing_sort<<< num_blocks, threads_per_block>>>( d_data, switch_at, n );

    // Perform the global bitonic merge stages for arrays larger than 1024 elements
    for (std::size_t k = 2048; k <= n; k <<= 1) {
        for (std::size_t stride = k >> 1; stride > 0; stride >>= 1) {
            bitonic_global_stage<<< num_blocks, threads_per_block>>>(
                d_data, n, k, stride );
        }
    }

    // Flip the last n/4 elements to descending (n<=1024 already flipped inside of block)
    if (n > 1024) {
        std::size_t suffix_size = n - switch_at;
        std::size_t number_of_pairs = suffix_size / 2;

        if (number_of_pairs > 0) {
            std::size_t reverse_threads = 256;
            std::size_t reverse_blocks = (number_of_pairs + reverse_threads - 1) / reverse_threads;

            reverse_suffix<<< reverse_blocks, reverse_threads>>>(d_data, switch_at, n );
        }
    }
    auto const kernel_end = std::chrono::high_resolution_clock::now();
    CHECK_ERROR("Executing kernel on device");

    // After the timer ends, copy the result back, free the device vector,
    // and echo out the timings and the results.
    cudaMemcpy( data.data(), d_data, sizeof( element_t ) * n, cudaMemcpyDeviceToHost );
    CHECK_ERROR("Transferring result back to host");
    cudaFree( d_data );
    CHECK_ERROR("Freeing device memory");

    std::cout << "GPU Solution time: "
              << std::chrono::duration_cast<std::chrono::nanoseconds>(kernel_end - kernel_start).count()
              << " ns" << std::endl;

    for( auto const x : data ) std::cout << x << " "; std::cout << std::endl;
}


} // namespace gpu
} // namespace a1
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "main.cu"
/**
 * Driver for the benchmark comparison. Generates random data,
 * runs the CPU baseline, and then runs your code.
 */

#include <cstddef>  // std::size_t type
#include <iostream> // std::cout, std::endl
#include <vector>

#include "algorithm_choices.h"
#include "data_generator.h"
#include "data_types.h"
#include "cuda_common.h"

int main()
{
    std::size_t const n = 1024*1024;
    std::size_t const switch_at = 3 * ( n >> 2 ) ;

    auto data = csc485b::a1::generate_uniform< element_t >( n );
    csc485b::a1::cpu::run_cpu_baseline( data, switch_at, n );
    csc485b::a1::gpu::run_gpu_soln( data, switch_at, n );
    csc485b::a1::gpu::run_gpu_radix_sort(data, n);

    return EXIT_SUCCESS;
}

In [ ]:

%cuda_group_run --group "source" --compiler-args "-O3 -g -std=c++20 -arch=sm_75"

CPU Baseline time: 87893430 ns
4 5 6 9 12 13 16 17 28 31 33 35 38 39 41 43 43 44 45 50 51 52 53 61 64 65 65 65 67 70 71 72 72 74 75 75 83 83 86 86 89 89 91 93 98 106 107 113 116 120 123 123 124 125 129 131 133 135 138 139 139 140 140 140 141 144 144 145 147 147 150 152 152 154 155 156 158 158 159 161 163 163 166 166 169 169 170 172 173 173 174 175 176 178 179 181 184 186 191 193 194 201 202 202 203 204 205 212 213 214 216 218 219 225 226 228 230 231 235 235 238 238 239 240 242 242 243 244 245 247 250 257 260 262 262 263 263 264 266 270 274 280 282 284 285 287 288 293 298 300 301 304 305 305 309 313 315 319 321 329 330 331 331 332 333 334 335 338 339 340 344 344 344 347 352 353 355 356 361 362 366 367 368 370 370 372 372 372 374 375 375 377 379 379 381 381 384 394 396 396 397 397 400 403 407 408 409 410 410 410 414 414 415 417 418 419 419 421 427 428 429 430 430 433 433 436 438 439 441 441 442 443 448 449 451 455 460 460 462 463 464 465 470 472 473 474 475 477 482 483 483 484 486 489 49